In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

df1 = pd.read_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\processed\df_literacy.csv", index_col = 0)
df2 = pd.read_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\processed\df_illiteracy.csv", index_col = 0)
df3 = pd.read_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\processed\df_gdp_schooling.csv", index_col = 0)

# Base Features

## GDP per Schooling_year

In [2]:
df3["gdp_per_schooling_year"] = df3["gdp"] / df3["avg_schooling_years"]
df3["gdp_per_schooling_year"].describe()

count     4919.000000
mean      2400.657736
std       2359.657034
min        115.727766
25%        842.095145
50%       1564.127784
75%       3179.216401
max      17209.310875
Name: gdp_per_schooling_year, dtype: float64

In [3]:
print(df3["gdp_per_schooling_year"].skew())

# Since skewness > 1.8, applied log_transformation
df3["log_gdp_per_schooling_year"] = np.log(df3["gdp_per_schooling_year"])
df3["log_gdp_per_schooling_year"].describe()

2.1935450184352154


count    4919.000000
mean        7.377883
std         0.918060
min         4.751241
25%         6.735893
50%         7.355084
75%         8.064390
max         9.753206
Name: log_gdp_per_schooling_year, dtype: float64

## Literacy Gender Gap (absolute)

In [4]:
df1["literacy_gender_gap_abs"] = df1["youth_literacy_rate_M"] - df1["youth_literacy_rate_F"]
df1["literacy_gender_gap_abs"].describe()

count    1103.000000
mean        3.373743
std         8.578952
min       -33.199700
25%        -0.600000
50%         0.000000
75%         4.000000
max        52.210000
Name: literacy_gender_gap_abs, dtype: float64

Median = 0 - In at least 50% of observations, youth male and female literacy are equal.

Mean = 3.37 - On average, male literacy is 3.37 percentage points higher than female.

Min = -33 - Female literacy 33 points higher than male.

Max = 52 - Severe gender inequality.

## Gender Gap %

In [5]:
df1["gender_gap_pct"] = (df1["literacy_gender_gap_abs"] / df1["youth_literacy_rate_M"]) * 100
df1["gender_gap_pct"].describe()

count    1103.000000
mean        5.009880
std        13.015350
min       -51.242013
25%        -0.610377
50%         0.000000
75%         4.325462
max        70.454545
Name: gender_gap_pct, dtype: float64

Median = 0 - At least 50% of observations show gender parity in youth literacy.

Mean = 5 - Female youth literacy is 5% lower than male literacy (relative).

STD = 13 - Inequality intensity varies dramatically across countries.

Min = -51 - Female literacy much higher than male

Max = +70 - Female literacy extremely low relative to male.

## Youth Literacy Average

In [6]:
df1["youth_literacy_avg"] = (df1["youth_literacy_rate_M"] + df1["youth_literacy_rate_F"]) / 2
df1["youth_literacy_avg"].describe()

count    1103.000000
mean       89.678806
std        15.850923
min        17.500000
25%        86.250000
50%        97.550000
75%        99.000000
max       100.000000
Name: youth_literacy_avg, dtype: float64

Median = 97.55 - More than 50% of country-year observations have youth literacy above ~97%.

Mean < Median (89.7 < 97.6) - A few very low-literacy countries are pulling the average down.

STD = 15.85 - Large disparity across countries.

Minimum = 17.5 - Fragile states, Conflict-affected regions and Low-income countries.

## Regional Standard Deviation

In [7]:
# This measures how unequal literacy is within each continent per year.
regional_std = (
    df1.groupby(["continent", "year"])["youth_literacy_avg"]
    .std()
    .reset_index(name = "regional_literacy_std")
)
regional_std

,continent,year,regional_literacy_std
0,Africa,1990,18.034111
1,Africa,1991,24.881385
2,Africa,1992,38.537320
3,Africa,1993,53.386562
4,Africa,1994,32.787193
...,...,...,...
161,South America,2019,0.402965
162,South America,2020,0.337682
163,South America,2021,0.257737
164,South America,2022,0.246644


In [8]:
df1 = df1.merge(regional_std, on = ["continent", "year"], how = "left")
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1103 entries, 0 to 1102
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   country                             1103 non-null   object 
 1   code                                1103 non-null   object 
 2   year                                1103 non-null   int64  
 3   adult_literacy_rate                 1103 non-null   float64
 4   youth_literacy_rate_M               1103 non-null   float64
 5   youth_literacy_rate_F               1103 non-null   float64
 6   continent                           1103 non-null   object 
 7   invalid_literacy_flag               1103 non-null   bool   
 8   adult_literacy_rate_outlier_flag    1103 non-null   bool   
 9   youth_literacy_rate_M_outlier_flag  1103 non-null   bool   
 10  youth_literacy_rate_F_outlier_flag  1103 non-null   bool   
 11  literacy_gender_gap_abs             1103 no

# Advanced Strategic Features

## Growth Metrics

### Literacy Growth Rate(YoY %)

In [9]:
# measures how fast literacy is improving (or declining) year over year
df1 = df1.sort_values(["country", "year"])

In [10]:
df1["youth_literacy_yoy"] = (
    df1.groupby("country")["youth_literacy_avg"].pct_change() * 100
)
df1["youth_literacy_yoy"].describe()

count    945.000000
mean       2.254590
std       10.684503
min      -41.080000
25%       -0.008046
50%        0.101163
75%        1.538462
max      125.714286
Name: youth_literacy_yoy, dtype: float64

Median ≈ 0.10 - In a typical year, literacy improves very slowly.

Mean = 2.25 - Some countries experienced large jumps, pulling the average upward.

STD = 10.68% - There may be volatility or data jumps.

Extreme Max = 125 & Min = -41 - Outliers

In [11]:
# More Stable Feature
df1["youth_literacy_abs_change"] = (df1.groupby("country")["youth_literacy_avg"].diff())
df1["youth_literacy_abs_change"].describe()

count    945.000000
mean       1.139487
std        5.012410
min      -36.500000
25%       -0.007925
50%        0.100000
75%        1.460685
max       55.000000
Name: youth_literacy_abs_change, dtype: float64

### 5-Year Rolling Literacy Average

In [12]:
# Smooths short-term fluctuations and shows the underlying trend.
df1 = df1.sort_values(["country", "year"])

In [13]:
df1["youth_literacy_5yr_avg"] = (
    df1.groupby("country")["youth_literacy_avg"]
    .rolling(window = 5)
    .mean()
    .reset_index(level = 0, drop = True)
)
df1["youth_literacy_5yr_avg"].describe()

count    561.000000
mean      91.236967
std       14.322258
min       27.600000
25%       90.500000
50%       97.970000
75%       98.900000
max      100.000000
Name: youth_literacy_5yr_avg, dtype: float64

Median ≈ 97.97 - For half the country-years, 5-year smoothed youth literacy is almost universal.

Mean ≈ 91.24 - A smaller group of low-literacy countries is dragging the average down.

Std = 14.32 - Educational inequality across countries is still large.

Minimum = 27.6 - Some countries still have structurally low literacy.

Upper Quartile ≈ 98.9 - Many countries are in the maturity phase of literacy transition.

### Literacy Momentum Score

In [14]:
df1 = df1.sort_values(["country", "year"])

df1["literacy_momentum_5yr"] = (
    df1.groupby("country")["youth_literacy_5yr_avg"]
    .diff()
)
df1["literacy_momentum_5yr"].describe()

count    475.000000
mean       0.744544
std        1.458827
min       -4.500000
25%        0.026000
50%        0.150000
75%        0.622534
max        8.787988
Name: literacy_momentum_5yr, dtype: float64

Mean ≈ 0.74 - On average, the 5-year literacy average is increasing by ~0.74 percentage points per year.

Median ≈ 0.15 - Most countries are improving slowly.; A few countries are accelerating strongly.; Distribution is right-skewed.

25th Percentile ≈ 0.026 - These countries are near saturation (97–100% literacy).

Max = 8.79 - Some countries increased their 5-year rolling literacy average by almost 9 percentage points in a single year.

Min = -4.5 - The smoothed literacy trend declined.

### Literacy Stability Index (LSI)

In [15]:
# Measures how stable is a country’s literacy trajectory
# Compute Rolling STD
df1["literacy_5yr_std"] = (
    df1.groupby("country")["youth_literacy_5yr_avg"]
    .rolling(window = 5)
    .std()
    .reset_index(level = 0, drop = True)
)

In [16]:
# Convert to Stability Index
df1["literacy_stability_index"] = 1 / (1 + df1["literacy_5yr_std"])
df1["literacy_stability_index"].describe()

count    282.000000
mean       0.761498
std        0.220631
min        0.128351
25%        0.659315
50%        0.827328
75%        0.920618
max        1.000000
Name: literacy_stability_index, dtype: float64

Mean ≈ 0.76 - Globally, literacy systems are moderately to highly stable.

Median ≈ 0.83 - Most countries are quite stable, but a few unstable ones pull the mean down.

75th Percentile ≈ 0.92 - These are likely: High-income nations; Near-saturation literacy systems; Mature education structures

Minimum ≈ 0.13 - Some country-years had high structural volatility in literacy trends.

STD = 0.22 - There is noticeable variation in stability across countries.

## Equity & Convergence Metrics

### Gender Gap Acceleration Rate

In [17]:
# Measures whether : Is the gender gap closing faster? Or Is it widening faster?

# Compute Gender Gap Growth (YoY Change)
df1 = df1.sort_values(["country", "year"])

df1["gender_gap_change"] = (
    df1.groupby("country")["literacy_gender_gap_abs"]
    .diff()
)

In [18]:
# Compute Acceleration
df1["gender_gap_acceleration"] = (
    df1.groupby("country")["gender_gap_change"]
    .diff()
)
df1["gender_gap_acceleration"].describe()

count    799.000000
mean      -0.185451
std        9.233430
min      -74.440000
25%       -1.000000
50%        0.000000
75%        1.000000
max       95.630690
Name: gender_gap_acceleration, dtype: float64

Median = 0 - For most country-years, the acceleration is zero.

Mean ≈ -0.19 - Globally, there is a very mild tendency toward accelerating gap reduction.

Std = 9.23 - A small subset of country-years experienced dramatic inequality shifts.

Min = -74.44 - The gap began closing much faster suddenly.

Max = 95.63 - Gap started widening much faster.

### Regional Convergence Index

In [19]:
# Measures countries within a region becoming more similar in literacy over time.

# Compute Regional Dispersion Per Year
regional_dispersion = (
    df1.groupby(["continent", "year"])["youth_literacy_avg"]
    .std()
    .reset_index(name = "regional_literacy_std")
)

In [20]:
# Measure Change in Dispersion (Convergence)
regional_dispersion = regional_dispersion.sort_values(["continent", "year"])

regional_dispersion["dispersion_change"] = (
    regional_dispersion.groupby("continent")["regional_literacy_std"]
    .diff()
)
regional_dispersion["dispersion_change"]

0            NaN
1       6.847274
2      13.655934
3      14.849242
4     -20.599369
         ...    
161    -0.158652
162    -0.065283
163    -0.079945
164    -0.011093
165    -0.063159
Name: dispersion_change, Length: 166, dtype: float64

In [21]:
# Convergence Index
regional_dispersion["regional_convergence_index"] = ( - regional_dispersion["dispersion_change"])
regional_dispersion.describe()

,year,regional_literacy_std,dispersion_change,regional_convergence_index
count,166.000000,138.000000,117.000000,117.000000
mean,2007.662651,7.774130,0.258572,-0.258572
std,9.644313,9.621256,5.453438,5.453438
min,1990.000000,0.000000,-20.599369,-20.796098
25%,2000.000000,0.687554,-1.204047,-1.364938
50%,2008.000000,2.746350,-0.060976,0.060976
75%,2016.000000,13.691418,1.364938,1.204047
max,2023.000000,53.386562,20.796098,20.599369


Regional Literacy Std:

Median ≈ 2.75 - In a typical continent-year, countries are quite similar in literacy.

Mean > median - A few continent-years have very high inequality.

Max = 53 - At some point, one region had extreme disparity.

Dispersion Change:

Median ≈ -0.06 - Slight typical convergence.

Mean ≈ +0.26 - On average, dispersion slightly increases.

Regional Convergence Index:

Median RCI ≈ +0.06 - Regions slowly become more similar.

Mean RCI ≈ -0.26 - Convergence is gradual and steady; Divergence is rare but sharp.

In [22]:
df1 = df1.merge(
    regional_dispersion[["continent", "year", "regional_convergence_index"]],
    on = ["continent", "year"],
    how = "left"
)


## Composite Metrics

### Education Development Index (EDI)

In [23]:
# EDI requires Literacy and Avg schooling years, so inner merge df1 and df3

# Merge literacy + schooling data
df_edi = df1.merge(
    df3[["country", "year", "avg_schooling_years"]],
    on = ["country", "year"],
    how = "inner"
)
df_edi.shape

(896, 25)

In [24]:
# Remove Invalid Rows
df_edi = df_edi[(~df_edi["invalid_literacy_flag"]) & (df_edi["avg_schooling_years"].notna())]

In [25]:
# Check Scale
df_edi[["adult_literacy_rate", "youth_literacy_avg", "avg_schooling_years"]].describe()

,adult_literacy_rate,youth_literacy_avg,avg_schooling_years
count,896.000000,896.000000,896.000000
mean,82.688474,90.893139,7.822037
std,17.534195,13.741902,2.354625
min,14.000000,20.000000,1.040000
25%,74.000000,88.779252,6.304500
50%,90.075850,97.500000,8.017000
75%,95.000000,99.000000,9.523000
max,100.000000,100.000000,13.060000


In [26]:
# Normalize Variables(Min Max Scaling)
scaler = MinMaxScaler()
cols = ["adult_literacy_rate", "youth_literacy_avg", "avg_schooling_years"]

df_edi[[i + '_norm' for i in cols]] = scaler.fit_transform(df_edi[cols])

In [27]:
# Create Literacy Index with weighted combination Adult(60%) & Young(40%)
df_edi["literacy_index"] = (
    0.6 * df_edi["adult_literacy_rate_norm"] +
    0.4 * df_edi["youth_literacy_avg_norm"]
)

In [28]:
# Create Final Education Development Index with Literacy(70%) & Schooling(30%)
df_edi["education_development_index"] = (
    0.7 * df_edi["literacy_index"] +
    0.3 * df_edi["avg_schooling_years_norm"]
)

In [29]:
df_edi["education_development_index"].describe()

count    896.000000
mean       0.752850
std        0.182895
min        0.010682
25%        0.675311
50%        0.820455
75%        0.876136
max        0.990233
Name: education_development_index, dtype: float64

Mean = 0.75 - Globally, most countries fall into moderate-to-high education development levels.

Median = 0.82 - This indicates Distribution is left-skewed (negatively skewed) & A few very low-performing countries are pulling the mean downward.

STD = 0.18 - There is noticeable inequality between countries in education development.

Min = 0.0106 - Indicates extreme underdevelopment in education in certain regions, likely low-income or conflict-affected countries.

### Education Spend Efficiency Score

In [30]:
# Answers - How effectively does economic capacity translate into education outcomes?

# Merging EDI with logged GDP
df_eff = df_edi.merge(
    df3[["country", "year", "log_gdp"]],
    on = ["country", "year"],
    how = "inner"
)

In [31]:
# Normalize GDP
df_eff["log_gdp_norm"] = scaler.fit_transform(df_eff[["log_gdp"]])

In [32]:
# Create Efficiency Score
df_eff["education_spend_efficiency"] = (
    df_eff["education_development_index"] - df_eff["log_gdp_norm"]
)

In [33]:
df_eff["education_spend_efficiency"].describe()

count    882.000000
mean       0.243400
std        0.132806
min       -0.188936
25%        0.179759
50%        0.253595
75%        0.323539
max        0.777922
Name: education_spend_efficiency, dtype: float64

Mean = 0.243 - Countries tend to perform better in education relative to their normalized GDP.

Median = 0.253 - Some strong negative underperformers pulling the mean down.

STD = 0.132 - There is meaningful inequality in how efficiently countries convert economic strength into educational outcomes.

Min = -0.1889 - Countries where GDP is relatively high but education outcomes lag behind.

Max = 0.7779 - Country achieves very high education development despite relatively low GDP.

### Illiteracy Burden Index

In [34]:
# Normalize illiteracy rate since it is already in %.
df2['illiteracy_rate_norm'] = scaler.fit_transform(df2[['illiteracy_rate']])

In [35]:
# Define Illiteracy Burden Index
df2['illiteracy_burden_index'] = df2['illiteracy_rate_norm']

In [36]:
df2["illiteracy_burden_index"].describe()

count    833.000000
mean       0.201610
std        0.222663
min        0.000000
25%        0.044944
50%        0.101124
75%        0.308459
max        1.000000
Name: illiteracy_burden_index, dtype: float64

Mean = 0.20 - Most countries have moderate-to-low illiteracy burden.

Median = 0.10 - A smaller group of high-burden countries pull the average upward. Most countries have very low burden.

STD = 0.22 - Significant disparity between low-burden and high-burden countries.

### Normalized Strategic Feature Set

In [37]:
strategic_features = [
    'education_development_index',
    'education_spend_efficiency',
    'illiteracy_burden_index',
    'gender_gap_pct',
    'log_gdp_norm',
    'regional_convergence_index'
]

df_strategic = df_edi.merge(
    df_eff[['country','year',
            'education_spend_efficiency',
            'log_gdp_norm']],
    on=['country','year'],
    how='inner'
)

df_strategic = df_strategic.merge(
    df2[['country','year','illiteracy_burden_index']],
    on=['country','year'],
    how='inner'
)

df_strategic = df_strategic[['country','year'] + strategic_features]

df_strategic.isnull().sum()

df_strategic = df_strategic.dropna()

In [38]:
df_strategic_norm = df_strategic.copy()

df_strategic_norm[[i + '_norm' for i in strategic_features]] = scaler.fit_transform(
    df_strategic[strategic_features]
)

# Save Final Dataset

In [39]:
df1.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\df_literacy.csv")

In [40]:
df2.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\df_illiteracy.csv")

In [41]:
df3.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\df_gdp_schooling.csv")

In [42]:
df_edi.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\df_edi.csv")

In [43]:
df_eff.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\df_eff.csv")

In [44]:
df_strategic_norm.to_csv(r"F:\DATA SCIENCE\Projects\Global Literacy & Educational Trends\data\featured\strategic_feature_master.csv",
    index=False
)

In [45]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1103 entries, 0 to 1102
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   country                             1103 non-null   object 
 1   code                                1103 non-null   object 
 2   year                                1103 non-null   int64  
 3   adult_literacy_rate                 1103 non-null   float64
 4   youth_literacy_rate_M               1103 non-null   float64
 5   youth_literacy_rate_F               1103 non-null   float64
 6   continent                           1103 non-null   object 
 7   invalid_literacy_flag               1103 non-null   bool   
 8   adult_literacy_rate_outlier_flag    1103 non-null   bool   
 9   youth_literacy_rate_M_outlier_flag  1103 non-null   bool   
 10  youth_literacy_rate_F_outlier_flag  1103 non-null   bool   
 11  literacy_gender_gap_abs             1103 no

In [46]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 833 entries, 2 to 2058
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       833 non-null    object 
 1   code                          833 non-null    object 
 2   year                          833 non-null    int64  
 3   illiteracy_rate               833 non-null    float64
 4   literacy_rate                 833 non-null    float64
 5   invalid_literacy_flag         833 non-null    bool   
 6   literacy_rate_outlier_flag    833 non-null    bool   
 7   illiteracy_rate_outlier_flag  833 non-null    bool   
 8   illiteracy_rate_norm          833 non-null    float64
 9   illiteracy_burden_index       833 non-null    float64
dtypes: bool(3), float64(4), int64(1), object(2)
memory usage: 54.5+ KB


In [47]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4963 entries, 25 to 11112
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     4963 non-null   object 
 1   code                        4963 non-null   object 
 2   year                        4963 non-null   int64  
 3   gdp                         4919 non-null   float64
 4   continent                   4963 non-null   object 
 5   avg_schooling_years         4963 non-null   float64
 6   invalid_gdp_flag            4963 non-null   bool   
 7   invalid_schooling_flag      4963 non-null   bool   
 8   log_gdp                     4919 non-null   float64
 9   log_gdp_zscore              4919 non-null   float64
 10  gdp_outlier_flag            4963 non-null   bool   
 11  gdp_per_schooling_year      4919 non-null   float64
 12  log_gdp_per_schooling_year  4919 non-null   float64
dtypes: bool(3), float64(6), int64(1), ob